# 03 · Integration of OpenTME and TCGA molecular data

This notebook joins the patient-level OpenTME table (notebook 01) with the TCGA molecular working table (notebook 02) into a single table with **one row per patient**. All downstream analyses (notebooks 04–07) read this table.

- **Cohort:** the OpenTME patients that passed image QC define the cohort. Patients absent from the molecular table (redacted in TCGA, excluded in notebook 02) are removed.
- **Missing values** are kept: a patient without, for example, mutation data has `NaN` in the `mut_` columns.
- **Provenance:** every column is documented with its modality (OpenTME / TCGA), source, label and coverage.

**Outputs**
- `<data>/processed/integrated_<opentme>.parquet`: the integrated table.
- `results/<indication>/03_integration/provenance.csv`: one row per column.

## 0. Setup

In [1]:
import pandas as pd
from IPython.display import display

from opentme_tcga import integration as integ
from opentme_tcga.config import ensure_dirs, load_config

INDICATION = "breast"          # name of a file in configs/indications/

cfg = load_config(INDICATION)
P = cfg["paths"]
SUBTYPE_COL = cfg["tcga_molecular"]["subtypes"]["column"]

ensure_dirs(cfg)
OUT_DIR = P["results"] / "03_integration"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Indication: {INDICATION} ({cfg['indication']['tcga_project']})")

Indication: breast (TCGA-BRCA)


## 1. Load and check both tables

Both tables must have one row per patient and well-formed 12-character barcodes (`TCGA-XX-XXXX`); the notebook stops otherwise.

In [2]:
opentme = integ.load_patient_table(P["opentme_patients"], key="TCGA_CASE_ID")
molecular = integ.load_patient_table(P["tcga_working"])          # barcode in the index

pd.DataFrame([integ.check_patients(opentme, "OpenTME (notebook 01)"),
              integ.check_patients(molecular, "TCGA molecular (notebook 02)")])

,table,rows,patients
0,OpenTME (notebook 01),1030,1030
1,TCGA molecular (notebook 02),1025,1025


## 2. Cohort and merge

OpenTME patients absent from the molecular table are excluded (the redacted TCGA patients of notebook 02). The remaining patients are joined one-to-one; a column name present in both tables would stop the notebook, so no variable is silently renamed.

In [3]:
integrated, excluded = integ.integrate(opentme, molecular)

print(f"OpenTME patients:                    {opentme['patient'].nunique()}")
print(f"Molecular patients:                  {molecular['patient'].nunique()}")
print(f"Excluded (not in molecular table):   {len(excluded)} {excluded}")
print(f"Molecular patients without OpenTME:  {len(set(molecular['patient']) - set(opentme['patient']))}")
print(f"Integrated table: {integrated.shape[0]} patients x {integrated.shape[1] - 1} variables")

OpenTME patients:                    1030
Molecular patients:                  1025
Excluded (not in molecular table):   5 ['TCGA-A7-A0DC', 'TCGA-AR-A0U1', 'TCGA-BH-A0HL', 'TCGA-BH-A0HN', 'TCGA-E9-A1NC']
Molecular patients without OpenTME:  0
Integrated table: 1025 patients x 755 variables


## 3. Provenance and coverage

Each column is documented with its modality, its source (OpenTME table from notebook 01, or molecular source from the notebook 02 dictionary), its level and label, and the number of patients with data. The coverage of the key variables of the immune analyses is shown first (`integration.key_variables` in the configuration, plus the subtype column).

In [4]:
source_file = P["results"] / "01_opentme" / "feature_source.csv"       # written by notebook 01
feature_source = pd.read_csv(source_file).set_index("feature")["table"] if source_file.exists() else None
if feature_source is None:
    print(f"{source_file.name} not found: run notebook 01 to record the OpenTME table of each feature.")
dictionary = pd.read_csv(P["tcga_dictionary"])
prov = integ.provenance(integrated, opentme.columns, feature_source, dictionary)

display(prov.groupby("modality").agg(n_variables=("column", "size"),
                                     median_pct_with_data=("pct_with_data", "median")))
key = [v for v in cfg["integration"]["key_variables"] + [SUBTYPE_COL] if v in set(prov["column"])]
prov.set_index("column").loc[key]

,n_variables,median_pct_with_data
modality,,
opentme,646,100.0
tcga,109,98.8


,modality,source,level,label,n_with_data,pct_with_data
column,,,,,,
CELL_DENSITY_LYMPHOCYTE_IN_CARCINOMA,opentme,OpenTME tme,NaN,NaN,1025,100.0
CELL_DENSITY_LYMPHOCYTE_IN_STROMA,opentme,OpenTME tme,NaN,NaN,1025,100.0
thor_leukocyte_fraction,tcga,Thorsson et al. 2018 (Table S1),main,Leukocyte fraction (Thorsson),998,97.4
thor_til_fraction_he,tcga,Thorsson et al. 2018 (Table S1),main,"TIL fraction, H&E deep learning (Saltz)",916,89.4
abs_purity,tcga,ABSOLUTE (PanCanAtlas),main,Tumour purity (ABSOLUTE),957,93.4
clin_tnbc,tcga,"Thennavan et al., Cell Genomics 2021, Data S1",main,Triple-negative (TNBC),973,94.9
clin_pam50,tcga,"Thennavan et al., Cell Genomics 2021, Data S1",main,PAM50 subtype,1023,99.8


In [5]:
# Molecular variables with the lowest coverage
prov[prov["modality"] == "tcga"].nsmallest(10, "pct_with_data")[["column", "source", "label", "n_with_data", "pct_with_data"]]

,column,source,label,n_with_data,pct_with_data
673,thor_bcr_richness,Thorsson et al. 2018 (Table S1),BCR richness (Thorsson),738,72.0
706,path_cell_cycle,Sanchez-Vega et al. 2018 (Table S4),Cell cycle pathway altered,743,72.5
721,aneu_8q_gain,Taylor 2018 via cBioPortal,8q gain,785,76.6
723,aneu_13q_loss,Taylor 2018 via cBioPortal,13q loss,824,80.4
717,aneu_1q_gain,Taylor 2018 via cBioPortal,1q gain,828,80.8
716,aneu_16q_loss,Taylor 2018 via cBioPortal,16q loss,830,81.0
722,aneu_22q_loss,Taylor 2018 via cBioPortal,22q loss,830,81.0
719,aneu_8p_loss,Taylor 2018 via cBioPortal,8p loss,853,83.2
724,aneu_20q_gain,Taylor 2018 via cBioPortal,20q gain,857,83.6
718,aneu_17p_loss,Taylor 2018 via cBioPortal,17p loss,875,85.4


## 4. Save

In [6]:
integrated.to_parquet(P["integrated"], index=False)
prov.to_csv(OUT_DIR / "provenance.csv", index=False)
print(f"Saved {P['integrated'].name}: {integrated.shape[0]} patients x {integrated.shape[1]} columns")
print(f"Saved provenance of {len(prov)} variables to {(OUT_DIR / 'provenance.csv').relative_to(P['root'])}")

Saved integrated_breast_cancer.parquet: 1025 patients x 756 columns
Saved provenance of 755 variables to results/breast/03_integration/provenance.csv
